In [5]:
function get_energy(id)
    if id == "ZERO"
        return "cc_fsol"
    else
        return "cc_fsol_twasp"
    end
end

function get_base_persistence_weights(id)
    x = Dict(
        "ONE" => [1.0, -0.2, -1.5],
        "TWO" => [1.0, -0.3, -1.2],
        "THREE" => [1.0, -0.4, -0.9],
        "FOUR" => [1.0, -0.7, -0.0],
        "FIVE" => [1.0, -0.7, -0.5],
        "SIX" => [1.0, -0.7, -0.7],
        "SEVEN" => [1.0, 0.0, 0.0],
        "EIGHT" => [1.0, 0.0, -2.0]
    )
    x[id]
end

get_base_persistence_weights (generic function with 1 method)

In [10]:
function get_perturbation_and_initialization(energy, mol_type, n_mol)
    if mol_type == "hard_sphere"
        return "single_random_only_translation", "random_only_translation"
    else
        if occursin("cc", energy) && n_mol > 2
            return "single_random_get_index", "random"
        else 
            return "single_random", "random"
        end
    end
end

mol_type = "6r7m"

x = "Vector{Float64}([])"
delaunay_eps = 100.0
overlap_jump = 0.0
overlap_slope = 1.1
rs = 1.4
η = 0.3665
exact_delaunay = mol_type == "6r7m" ? "false" : "true"

T_search_runs = 0
T_search_time = 0.0

simulation_time_minutes = 12 * 60.0
algorithm = "rwm"

#sa_level = "[1.0,0.8,0.6,0.4,0.2]"
sa_level = "[0.0]"

n_mol = 2

id = "EIGHT"
energy = get_energy(id)

bnds = 100.0

perturbation, initialization = get_perturbation_and_initialization(energy, mol_type, n_mol)

σ_r = 0.3
σ_t = 1.25

1.25

In [11]:
temperature = 4.5
for pw_scale in 0.1:0.05:0.5
    persistence_weights = get_base_persistence_weights(id)
    persistence_weights = persistence_weights .* pw_scale
    persistence_weights = "[$(persistence_weights[1]),$(persistence_weights[2]),$(persistence_weights[3])]" 
    scale_string = "$(id)_$(Int(round(pw_scale * 1000)))"
    comment = "$(Int(round(temperature * 10)))"
    comment = replace(comment, " " => "_")
    simulations_per_combination = 40

    input_specifier = "$(algorithm)_$(scale_string)_$(comment)"
    output_directory = "../Simulations/unsorted_output/$(input_specifier)/"

    open("../configs/$(input_specifier)_config.txt", "w") do io
        i = 1
        println(io,"ArrayTaskID input_string")
        output_directory = "../Simulations/unsorted_output/$(input_specifier)/"
        for _ in 0:simulations_per_combination-1
            name = "$(i)"
            input_string = escape_string("name=\"$name\";x=$(x);temperature=$(temperature);alg=\"$(algorithm)\";sa_level=$(sa_level);T_search_runs=$(T_search_runs);T_search_time=$(T_search_time);rs=$rs;η=$η;σ_t=$σ_t;σ_r=$σ_r;overlap_jump=$overlap_jump;overlap_slope=$overlap_slope;bnds=$bnds;persistence_weights=$persistence_weights;n_mol=$n_mol;mol_type=\"$mol_type\";nrg=\"$energy\";prtbt=\"$perturbation\";intl=\"$initialization\";output_directory=\"$output_directory\";delaunay_eps=$delaunay_eps;exact_delaunay=$exact_delaunay;comment=\"$comment\";simulation_time_minutes=$simulation_time_minutes")
            println(io, "$i $input_string")
            i += 1
        end
    end
    total_simulations = length(readlines("../configs/$(input_specifier)_config.txt")) - 1

    total_time_needed = simulation_time_minutes + (T_search_runs * T_search_time)

    hours = Int(round(total_time_needed / 60.0))
    days = hours ÷ 24
    remaining_hours = hours % 24
    remaining_hours_string = remaining_hours < 10 ? "0$(remaining_hours)" : string(remaining_hours)
    buffer_time_string = total_time_needed < 5 ? "0$(Int(round(total_time_needed))+2)" : "30"

    open("../$(input_specifier)_script.job", "w") do io
        println(io, "#!/bin/bash")
        println(io, "#SBATCH --job-name=SolvationSimulations")
        println(io, "#SBATCH --time=0$(days)-$(remaining_hours_string):$(buffer_time_string)")
        println(io, "#SBATCH --ntasks=1")
        println(io, "#SBATCH --cpus-per-task=1")
        println(io, "#SBATCH --mem-per-cpu=1G")
        println(io, "#SBATCH --array=1-$(total_simulations)")
        println(io, "#SBATCH --chdir=/work/spirandelli/MorphoMolHPC/")
        println(io, "#SBATCH -o ./job_log/$(input_specifier)/%a.out # STDOUT")
        println(io, "")
        println(io, "export http_proxy=http://proxy2.uni-potsdam.de:3128 #Setting proxy, due to lack of Internet on compute nodes.")
        println(io, "export https_proxy=http://proxy2.uni-potsdam.de:3128")
        println(io, "")
        println(io, "module purge")
        println(io, "source ../oineus_venv/bin/activate")    
        println(io, "module load devel/CMake/3.27.6-GCCcore-13.2.0")
        println(io, "module load devel/Boost/1.83.0-GCC-13.2.0")
        println(io, "module load lang/Julia/1.7.3-linux-x86_64")    
        println(io, "")
        println(io, "# Specify the path to the config file")
        println(io, "config=hpc_scripts/configs/$(input_specifier)_config.txt")
        println(io, "")
        println(io, "# Extract the variables from config file")
        println(io, "config_string=\$(awk -v ArrayTaskID=\$SLURM_ARRAY_TASK_ID '\$1==ArrayTaskID {print \$2}' \$config)")
        println(io, "")
        println(io, "julia -e \"$(escape_string("include(\"julia_scripts/generic_call.jl\"); generic_call(\"\$config_string\")"))\"")
    end
end